##*ADEOTI Nihimath*
##Cours: LLM

*Projet: Système de RAG sur la documentation web de MDN en français et comparer:
- **closed-book** (le LLM seul) vs **RAG** (LLM + récupération) ;
- l'effet du nombre de passages **k** ;
- un retrieveur **de base** vs **spécialisé** (fine-tuné).



## 0. Commençons par s'assurer de travailler avec le GPU

In [ ]:
!nvidia-smi

La commande ayant fonctionné, on est bien sur le GPU T4.

## 1. Récupérer le code et installer les dépendances
Remplacez l'URL par celle de **votre** dépôt après l'avoir poussé sur GitHub.

In [ ]:
!git clone https://github.com/nihmad/systeme-rag.git
%cd systeme-rag
!pip install -q -r requirements.txt

## 2. Construire le corpus (clone partiel de MDN + nettoyage + découpage)

In [ ]:
import gc, torch

# supprime toutes les variables qui pointent vers un modèle
for name in ["model", "generator", "gen", "llm", "pipe"]:
    if name in globals():
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()

print(f"VRAM libre : {torch.cuda.mem_get_info()[0]/1e9:.1f} Go")

In [ ]:
!python scripts/01_build_corpus.py

Aperçu de quelques passages :

In [ ]:
import json
chunks = [json.loads(l) for l in open('data/chunks.jsonl', encoding='utf-8')]
print('Nombre de passages :', len(chunks))
print(chunks[0]['title'])
print(chunks[0]['text'][:300], '…')

## 3. Indexer les passages (embeddings + FAISS)

In [ ]:
!python scripts/02_build_index.py

## 4. Générer le jeu d'évaluation
On crée des paires (question, réponse, passage source) à partir du corpus.
*La génération est lente : commencez avec un petit `-n` pour tester.*
Le fichier produit (`data/eval_set.jsonl`) est ensuite versionné dans le dépôt :
c'est lui qui rend vos résultats reproductibles.

In [ ]:
!python scripts/03_make_eval_set.py -n 10

In [ ]:
import json
exemples = [json.loads(l) for l in open('data/eval_set.jsonl', encoding='utf-8')]
for r in exemples[:3]:
    print("Q :", r['question'])
    print("R :", r['answer'])
    print("source :", r['gold_chunk_id'])

In [ ]:
!python scripts/03_make_eval_set.py -n 100

## 5. Évaluation : RAG vs closed-book + ablation sur k
Produit `results/report_base.json`.

In [ ]:
%cd /content/systeme-rag
!python scripts/04_run_evaluation.py --n_gen 30

## 6. Spécialiser le retrieveur, puis comparer
On fine-tune l'embedding model sur le domaine, on réindexe, et on réévalue
pour mesurer le gain de la spécialisation.

In [ ]:
!python scripts/05_finetune_embedder.py --epochs 2
!python scripts/02_build_index.py --finetuned
!python scripts/04_run_evaluation.py --finetuned --n_gen 30

## 7. Comparer les rapports
Tableau récapitulatif base vs spécialisé.

In [ ]:
import json, pandas as pd
rows = []
for tag in ['base', 'finetuned']:
    try:
        r = json.load(open(f'results/report_{tag}.json', encoding='utf-8'))
    except FileNotFoundError:
        continue
    d = {'index': tag, **r['retrieval']}
    if 'generation' in r:
        d['F1_RAG'] = r['generation']['rag']['F1']
        d['F1_closed_book'] = r['generation']['closed_book']['F1']
    rows.append(d)
pd.DataFrame(rows)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

os.makedirs("results", exist_ok=True)

# Données issues de tes rapports d'évaluation
metriques = ["hit@1", "hit@3", "hit@5", "hit@10", "MRR"]
base      = [0.55, 0.81, 0.88, 0.94, 0.6923]
finetuned = [0.63, 0.90, 0.93, 0.96, 0.7636]

x = np.arange(len(metriques))
largeur = 0.38

fig, ax = plt.subplots(figsize=(8, 4.5))
b1 = ax.bar(x - largeur/2, base,      largeur, label="Embedder de base",   color="#888780")
b2 = ax.bar(x + largeur/2, finetuned, largeur, label="Embedder fine-tuné", color="#1D9E75")

for barres in (b1, b2):
    ax.bar_label(barres, fmt="%.2f", padding=2, fontsize=9)

ax.set_ylabel("Score")
ax.set_title("Récupération : base vs fine-tuné")
ax.set_xticks(x, metriques)
ax.set_ylim(0, 1.05)
ax.legend()
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("results/retrieval_base_vs_finetuned.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
import sys; sys.path.insert(0, '.')
from ragdoc.retriever import Retriever
from ragdoc.generator import Generator
from ragdoc.pipeline import RagPipeline

retriever = Retriever.load()
generator = Generator()
rag = RagPipeline(retriever, generator)

out = rag.answer("À quoi sert l'élément HTML <article> ?")
print("RÉPONSE :", out['answer'])
print("\nSOURCES :")
for s in out['sources']:
    print(' -', s['title'], '|', round(s['score'], 3))

## 8. Analyse et conclusions



Sur la récupération, le fine-tuning de l'embedder gagne sur toute la ligne, avec les gains les plus nets en haut du classement : hit@1 passe de 0.55 à 0.63 (+8 points) et hit@3 de 0.81 à 0.90 (+9 points), tandis que le MRR grimpe de 0.69 à 0.76. C'est exactement le comportement recherché : le modèle spécialisé place le bon document plus souvent en première position ou tout en haut. Les gains se resserrent à hit@10 (0.94 → 0.96) à cause d'un effet plafond : retrouver le bon document dans le top-10 était déjà presque systématique, il restait peu de marge.
Sur la génération, le résultat central c'est l'écart entre RAG et closed-book. En base, le F1 passe de 0.144 (closed-book) à 0.312 (RAG), soit plus du double.**Autrement dit, donner le contexte récupéré au LLM améliore drastiquement la qualité des réponses par rapport à sa seule mémoire paramétrique.**

**Le retrieval n'est pas décoratif, il porte la performance.**
Le fine-tuning du retriever se répercute aussi sur la génération end-to-end, mais plus modestement : F1 0.312 → 0.325, ROUGE-L 0.296 → 0.303, et l'EM décolle légèrement de 0 à 0.033.  Un meilleur retrieval donne un meilleur contexte, donc de meilleures réponses, mais le gain est amorti parce qu'aux rangs effectivement utilisés (top-k) les deux retrievers ramènent déjà souvent le bon document donc l'amélioration ne joue que sur les cas limites.

L'EM est quasi nul partout. C'est normal et attendu. L'Exact Match exige que la réponse générée corresponde mot pour mot à la référence, or un LLM génératif comme Mistral produit des phrases complètes et reformulées, pas des spans exacts. Pour de la génération libre, l'EM est donc une métrique peu adaptée. Ce sont le F1 (recouvrement de tokens) et le ROUGE-L qui sont pertinents ici.
Une évaluation sémantique (BERTScore, ou un LLM-as-judge) capturerait mieux la qualité réelle des réponses.
En résumé : RAG double le F1 face au closed-book (la valeur ajoutée de la récupération) ; la spécialisation de l'embedder améliore surtout le retrieval (hit@1 +8 pts, MRR +7 pts) et, par ricochet, légèrement la génération ; et l'EML est inadapté à la génération libre, d'où l'intérêt de métriques de recouvrement et, en perspective, d'une évaluation sémantique.

In [ ]:
!pip install gradio -q

import gradio as gr

def repondre(question):
    if not question or not question.strip():
        return "Pose une question.", ""
    out = rag.answer(question)
    reponse = out["answer"]
    sources = "**Sources récupérées :**\n" + "\n".join(
        f"- {s['title']}  *(score : {round(s['score'], 3)})*"
        for s in out["sources"]
    )
    return reponse, sources

with gr.Blocks(title="Système RAG") as demo:
    gr.Markdown(
        "# Démonstrateur RAG\n"
        "Pose une question : le système récupère les documents pertinents "
        "puis génère une réponse avec Mistral-7B."
    )
    question = gr.Textbox(
        label="Ta question",
        placeholder="Ex. : À quoi sert l'élément HTML <article> ?",
    )
    btn = gr.Button("Interroger", variant="primary")
    reponse = gr.Textbox(label="Réponse générée", lines=6)
    sources = gr.Markdown()

    btn.click(repondre, inputs=question, outputs=[reponse, sources])
    question.submit(repondre, inputs=question, outputs=[reponse, sources])

    gr.Examples(
        examples=[
            "À quoi sert l'élément HTML <article> ?",
            "Quelle est la différence entre <div> et <span> ?",
            "Comment fonctionne l'attribut alt d'une image ?",
        ],
        inputs=question,
    )

demo.launch(share=True, debug=False)